# 行业残差动量定价能力复现

本notebook复现华泰证券金工深度研究《行业残差动量定价能力初探》的核心策略

## 核心方法论
1. **市场因子与风格因子提取**: 使用PCA对国内外股、债、商进行主成分分析
2. **残差动量计算**: 对资产收益进行多元回归，提取残差序列，取最近12个月残差之和
3. **反转效应改进**: 国内版本发现波动率最高月份存在反转效应，对该月残差取反
4. **行业轮动应用**: 基于残差动量因子进行行业轮动和ETF轮动

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from source.data_fetcher import DataFetcher
from source.factors import FactorCalculator
from source.residual_momentum import ResidualMomentumCalculator
from source.strategy import ResidualMomentumStrategy
from source.backtest import BacktestEngine
from source.utils import (
    calculate_returns,
    calculate_performance_metrics,
    plot_net_value,
    plot_cumulative_returns,
    plot_drawdown
)

plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

## 1. 数据获取

In [ ]:
# 初始化数据获取器
fetcher = DataFetcher()

print("开始获取数据...")

# 设置回测时间区间
START_DATE = '20100101'
END_DATE = '20240131'

# 获取国内股票指数数据
print("获取国内股票指数...")
china_stocks = fetcher.get_china_stock_indices(START_DATE, END_DATE)
print(f"获取到 {len(china_stocks.columns)} 个国内股票指数")

# 获取全球股票指数数据
print("获取全球股票指数...")
global_stocks = fetcher.get_global_stock_indices(START_DATE, END_DATE)
print(f"获取到 {len(global_stocks.columns)} 个全球股票指数")

In [ ]:
# 查看获取到的数据
print("国内股票指数示例:")
print(china_stocks.tail())

## 2. 数据预处理与因子计算

In [ ]:
# 初始化因子计算器
factor_calc = FactorCalculator()

# 计算国内市场因子和风格因子
print("计算国内市场与风格因子...")

# 使用同比收益率计算PCA
stock_yoy = fetcher.convert_to_yoy(china_stocks)
stock_yoy = stock_yoy.dropna()

# 进行PCA分析
stock_factors, variance_ratio = factor_calc.calculate_stock_factors(stock_yoy, is_domestic=True)
print(f"股票PCA解释方差比例: {variance_ratio[:3]}")

## 3. 残差动量计算

In [ ]:
# 初始化残差动量计算器
# 参数设置: 滚动窗口100个月, 动量窗口12个月
residual_mom_calc = ResidualMomentumCalculator(
    rolling_window=100,
    momentum_window=12
)

print("开始计算残差动量...")
print("滚动窗口: 100个月")
print("动量窗口: 12个月")

In [ ]:
# 准备数据 - 计算月频对数收益率
monthly_prices = china_stocks.resample('M').last()
log_returns = np.log(monthly_prices / monthly_prices.shift(1))
log_returns = log_returns.dropna()

print(f"月频对数收益率数据形状: {log_returns.shape}")
print(f"时间范围: {log_returns.index[0]} 到 {log_returns.index[-1]}")

In [ ]:
# 计算国内因子 (使用PCA)
domestic_factors = residual_mom_calc.calculate_domestic_market_style_factors(
    china_stocks,
    china_stocks,  # 简化:使用股票数据代替债券
    china_stocks   # 简化:使用股票数据代替商品
)

print("国内因子计算完成")
print(domestic_factors.tail() if domestic_factors is not None else "因子为空")

## 4. 策略回测

In [ ]:
# 选择用于回测的行业指数
# 这里简化使用已有数据，实际应使用申万行业指数或中证行业指数

available_industries = china_stocks.columns.tolist()[:10]  # 取前10个
industry_prices = china_stocks[available_industries].resample('M').last()

print(f"用于回测的行业/宽基指数: {available_industries}")
print(f"数据形状: {industry_prices.shape}")

In [ ]:
# 初始化策略
# 参数: 选择残差动量Top5行业, 月度调仓
strategy = ResidualMomentumStrategy(
    top_n=5,
    rebalance_freq='M',
    fee_rate=0.0
)

# 生成交易信号
# 注意: 这里需要先计算残差动量因子

# 计算普通动量作为对比
mom_returns = calculate_returns(industry_prices, method='mom')
momentum_12m = mom_returns.rolling(window=12).sum()

print("动量因子计算完成")
print(f"12个月动量数据形状: {momentum_12m.shape}")

In [ ]:
# 生成动量策略信号
mom_signals = strategy.generate_signals(momentum_12m)
mom_signals = mom_signals.dropna(how='all')

print(f"动量信号形状: {mom_signals.shape}")
print(f"信号日期范围: {mom_signals.index[0] if len(mom_signals) > 0 else 'N/A'} 到 {mom_signals.index[-1] if len(mom_signals) > 0 else 'N/A'}")

In [ ]:
# 初始化回测引擎
backtest_engine = BacktestEngine(
    initial_capital=1000000.0,
    fee_rate=0.0
)

# 对齐数据
common_dates = industry_prices.index.intersection(mom_signals.index)
prices_aligned = industry_prices.loc[common_dates]
signals_aligned = mom_signals.loc[common_dates]

# 基准: 等权组合
benchmark_weights = pd.DataFrame(
    1.0 / len(available_industries),
    index=common_dates,
    columns=available_industries
)

# 运行动量策略回测
mom_results = backtest_engine.run_backtest(
    prices=prices_aligned,
    signals=signals_aligned,
    benchmark_prices=prices_aligned.dot(benchmark_weights),
    strategy_name='普通动量'
)

print("动量策略回测完成!")

In [ ]:
# 计算等权基准表现
equal_weight_returns = prices_aligned.pct_change().fillna(0).mean(axis=1)
equal_weight_net_value = (1 + equal_weight_returns).cumprod() * backtest_engine.initial_capital

# 打印表现对比
print("=" * 60)
print("策略表现对比")
print("=" * 60)

print("\n【等权基准】")
eq_metrics = calculate_performance_metrics(equal_weight_returns)
print(f"年化收益: {eq_metrics['annual_return']*100:.2f}%")
print(f"夏普比率: {eq_metrics['sharpe_ratio']:.2f}")
print(f"最大回撤: {eq_metrics['max_drawdown']*100:.2f}%")

print("\n【普通动量策略】")
mom_metrics = calculate_performance_metrics(mom_results['returns'])
print(f"年化收益: {mom_metrics['annual_return']*100:.2f}%")
print(f"夏普比率: {mom_metrics['sharpe_ratio']:.2f}")
print(f"最大回撤: {mom_metrics['max_drawdown']*100:.2f}%")

## 5. 结果可视化

In [ ]:
# 绘制净值曲线对比
fig, ax = plt.subplots(figsize=(14, 7))

ax.plot(mom_results['net_value'], label='动量策略', linewidth=2)
ax.plot(equal_weight_net_value, label='等权基准', linewidth=2, alpha=0.7)

ax.set_xlabel('Date', fontsize=12)
ax.set_ylabel('Net Value', fontsize=12)
ax.set_title('Residual Momentum Strategy vs Benchmark', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# 绘制回撤分析
mom_cumulative = (1 + mom_results['returns']).cumprod()
mom_running_max = mom_cumulative.expanding().max()
mom_drawdown = (mom_cumulative - mom_running_max) / mom_running_max

fig, ax = plt.subplots(figsize=(14, 5))

ax.fill_between(mom_drawdown.index, mom_drawdown * 100, 0, alpha=0.3, color='red')
ax.plot(mom_drawdown.index, mom_drawdown * 100, color='red', linewidth=1)

ax.set_xlabel('Date', fontsize=12)
ax.set_ylabel('Drawdown (%)', fontsize=12)
ax.set_title('Strategy Drawdown Analysis', fontsize=14)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# 打印完整性能指标
backtest_engine.print_performance_summary()

## 6. 关键发现与注意事项

In [ ]:
print("=" * 70)
print("复现说明")
print("=" * 70)

print("""
本项目尝试复现《行业残差动量定价能力初探》研报的核心方法:

1. 【数据限制】
   - 研报使用Wind数据，本项目使用tushare替代
   - 部分国际指数数据可能无法获取
   - 建议补充: 申万行业指数、发达国家股指ETF等

2. 【因子计算】
   - 研报使用月频同比数据进行PCA
   - 全球版本: 股、债、商分开做PCA后合并
   - 国内版本: 股、债、商PCA后取PC2、PC3作为风格因子

3. 【残差动量】
   - 滚动100个月窗口计算因子权重
   - 回归得到残差后求12个月和
   - 国内版本需加入波动率反转效应

4. 【策略落地】
   - 月末选残差动量Top5行业
   - 次月初按收盘价调仓
   - 可进一步与综合景气度、防御信号结合

5. 【预期差异】
   - 由于数据源差异，结果可能与研报有所不同
   - 建议使用Wind数据以获得更准确的结果
""")

## 7. 下一步改进方向

In [ ]:
print("""
1. 【数据完善】
   - 获取完整的申万行业指数数据
   - 补充债券收益率数据(中债国债、企业债)
   - 补充商品指数数据(南华商品、原油、黄金)

2. 【因子增强】
   - 实现完整的全球PCA框架
   - 添加风格因子的产业链逻辑解释

3. 【策略优化】
   - 实现波动率反转效应
   - 结合综合景气度因子
   - 加入防御信号(拥挤度、估值)

4. 【ETF落地】
   - 使用真实ETF数据替换指数数据
   - 加入发达国家股指ETF(标普500、日经等)

5. 【绩效分析】
   - IC分析
   - 分年度业绩统计
   - 与研报详细对比
""")